# DermaCheck AI Clinical v4 - Enhanced Deployment\n\n**Version**: 4.0 (Clinical Research Integration)  \n**Date**: 2026-02-08  \n**Frameworks**: DermNet NZ + LearnDern + Fitzpatrick + SOCS + NHS e-LfH + CyberDerm\n\n## 📚 What's New in v4:\n\n### Clinical Prompt Library:\n- ✅ **Master Clinical Prompt** - Comprehensive LearnDerm 5-step + Fitzpatrick Wheel evaluation\n- ✅ **Melanoma Screening** - ABCDE criteria + Acral lentiginous melanoma detection\n- ✅ **Emergency Triage** - SJS/TEN, meningococcemia, necrotizing fasciitis red flags\n- ✅ **Skin of Color** - Fitzpatrick IV-VI specialized (erythema variations, PIH, ALM)\n\n### Features:\n- 🧠 Intelligent prompt selection based on presentation\n- 🎨 Fitzpatrick-aware color interpretation\n- 🚨 Emergency red flag detection\n- 📊 Deterministic generation (reproducible results)\n- ⚖️  Equity protocols (<5% disparity target)\n\n### Quality Targets:\n```\n✅ Top-1 Accuracy: >85%\n✅ Melanoma Sensitivity: >95%\n✅ Fitzpatrick Equity: <5% disparity\n✅ Emergency Detection: 100%\n```

## 🔧 Setup Instructions\n\n### IMPORTANT: Upload Prompt Library\n\nBefore running this notebook, you MUST upload the prompts folder as a Kaggle Dataset:\n\n1. **Create Kaggle Dataset**:\n   - Go to Kaggle → Your Profile → Datasets → New Dataset\n   - Upload entire `/prompts/` folder (5 files):\n     * master_clinical_prompt.txt\n     * melanoma_screening_prompt.txt\n     * emergency_triage_prompt.txt\n     * skin_of_color_prompt.txt\n     * README.md\n   - Name it: `dermacheck-clinical-prompts`\n   - Make it public or private\n\n2. **Add Dataset to Notebook**:\n   - In this notebook, click 'Add Data' → Search your username\n   - Select `dermacheck-clinical-prompts`\n   - It will be mounted at `/kaggle/input/dermacheck-clinical-prompts/`\n\n3. **Verify Path**:\n   - Run cell below to check prompts are accessible

In [ ]:
# Cell 1: Verify Prompt Library\nimport os\n\nPROMPTS_DIR = '/kaggle/input/dermacheck-clinical-prompts/'\n\nprint('📂 Checking prompt library...')\n\nif os.path.exists(PROMPTS_DIR):\n    files = os.listdir(PROMPTS_DIR)\n    print(f'✅ Prompts directory found!')\n    print(f'\nFiles ({len(files)}):')\n    for f in files:\n        size = os.path.getsize(os.path.join(PROMPTS_DIR, f))\n        print(f'  - {f} ({size:,} bytes)')\nelse:\n    print('❌ ERROR: Prompts directory not found!')\n    print('\n⚠️  CRITICAL: You must upload the prompts folder as a Kaggle dataset first!')\n    print('\nInstructions:')\n    print('1. Create new Kaggle dataset: dermacheck-clinical-prompts')\n    print('2. Upload all 5 files from /prompts/ folder')\n    print('3. Add dataset to this notebook (Add Data button)')\n    print('4. Re-run this cell')

## 📦 Install Dependencies

In [ ]:
# Cell 2: Install packages\n!pip install -q fastapi uvicorn python-multipart pyngrok nest-asyncio transformers>=4.50.0

## 🤖 Load MedGemma Model

In [ ]:
# Cell 3: Load MedGemma 1.5\nfrom transformers import AutoProcessor, AutoModelForImageTextToText\nimport torch\nimport os\n\nmodel_id = 'google/medgemma-1.5-4b-it'\n\nprint('🔄 Loading MedGemma 1.5-4b-it...')\n\nprocessor = AutoProcessor.from_pretrained(\n    model_id,\n    token=os.environ.get('HF_TOKEN')  # Set in Kaggle Secrets\n)\n\nmodel = AutoModelForImageTextToText.from_pretrained(\n    model_id,\n    token=os.environ.get('HF_TOKEN'),\n    torch_dtype=torch.bfloat16,\n    device_map='auto'\n)\n\nprint(f'✅ Model loaded on {model.device}')\nprint(f'📊 Model size: {model.num_parameters():,} parameters')

## 🚀 Deploy Clinical API v4

In [ ]:
# Cell 4: Deploy Enhanced Clinical Backend\n\nimport nest_asyncio\nnest_asyncio.apply()\n\nfrom fastapi import FastAPI, UploadFile, File, Form, HTTPException\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom pyngrok import ngrok\nimport uvicorn\nfrom io import BytesIO\nfrom PIL import Image\nfrom typing import Optional\nfrom enum import Enum\n\napp = FastAPI(title='DermaCheck AI Clinical v4')\n\n# CORS\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=['*'],\n    allow_credentials=True,\n    allow_methods=['*'],\n    allow_headers=['*'],\n)\n\n# ═══════════════════════════════════════════════════════════\n# LOAD CLINICAL PROMPTS\n# ═══════════════════════════════════════════════════════════\n\nPROMPTS_DIR = '/kaggle/input/dermacheck-clinical-prompts/'\n\ndef load_prompt(filename: str) -> str:\n    try:\n        with open(os.path.join(PROMPTS_DIR, filename), 'r') as f:\n            return f.read()\n    except FileNotFoundError:\n        print(f'⚠️  Warning: {filename} not found')\n        return ''\n\nprint('📚 Loading clinical prompt library...')\nMASTER_CLINICAL = load_prompt('master_clinical_prompt.txt')\nMELANOMA_SCREENING = load_prompt('melanoma_screening_prompt.txt')\nEMERGENCY_TRIAGE = load_prompt('emergency_triage_prompt.txt')\nSKIN_OF_COLOR = load_prompt('skin_of_color_prompt.txt')\nprint('✅ Prompts loaded!')\n\n# ═══════════════════════════════════════════════════════════\n# PROMPT SELECTION LOGIC\n# ═══════════════════════════════════════════════════════════\n\nclass PromptType(str, Enum):\n    EMERGENCY = 'emergency_triage'\n    MELANOMA = 'melanoma_screening'\n    SKIN_OF_COLOR = 'skin_of_color'\n    MASTER = 'master_clinical'\n\ndef select_prompt(fitzpatrick: int, fever: bool, rapid: bool, location: str, complaint: str):\n    # Priority 1: Emergency\n    if fever or rapid:\n        print('🚨 EMERGENCY TRIAGE')\n        return EMERGENCY_TRIAGE, PromptType.EMERGENCY\n    \n    # Priority 2: Melanoma\n    melanoma_kw = ['mole', 'spot', 'changing', 'pigmented', 'melanoma']\n    acral = ['palm', 'sole', 'nail', 'foot', 'hand']\n    mucosal = ['mouth', 'oral', 'genital']\n    \n    if (any(k in complaint.lower() for k in melanoma_kw) or \n        any(k in location.lower() for k in acral + mucosal)):\n        print('🔍 MELANOMA SCREENING')\n        return MELANOMA_SCREENING, PromptType.MELANOMA\n    \n    # Priority 3: Skin of color\n    if fitzpatrick >= 4:\n        print(f'🎨 SKIN OF COLOR (Fitz {fitzpatrick})')\n        return SKIN_OF_COLOR, PromptType.SKIN_OF_COLOR\n    \n    # Default: Master clinical\n    print('📋 MASTER CLINICAL')\n    return MASTER_CLINICAL, PromptType.MASTER\n\n# ═══════════════════════════════════════════════════════════\n# API ENDPOINTS\n# ═══════════════════════════════════════════════════════════\n\n@app.get('/')\nasync def root():\n    return {\n        'app': 'DermaCheck AI Clinical',\n        'version': 'v4-enhanced',\n        'model': 'MedGemma 1.5-4b-it',\n        'status': 'ready',\n        'prompts': ['master', 'melanoma', 'emergency', 'skin_of_color'],\n        'features': ['intelligent_selection', 'fitzpatrick_aware', 'deterministic']\n    }\n\n@app.post('/analyze')\nasync def analyze(\n    file: UploadFile = File(...),\n    age: int = Form(...),\n    sex: str = Form(...),\n    fitzpatrick_type: int = Form(...),\n    body_location: str = Form(...),\n    duration: str = Form(...),\n    chief_complaint: str = Form('Skin evaluation'),\n    symptoms: str = Form(''),\n    itch_score: int = Form(0),\n    pain_present: bool = Form(False),\n    warmth_present: bool = Form(False),\n    fever: bool = Form(False),\n    rapidly_progressive: bool = Form(False),\n    recent_medications: str = Form('None'),\n    known_allergies: str = Form('None'),\n    medical_history: str = Form('None'),\n    family_history: str = Form('None'),\n    recent_travel: str = Form('None')\n):\n    try:\n        print(f'\\n{'='*70}')\n        print(f'📸 ANALYSIS: {age}y {sex}, Fitz {fitzpatrick_type}, {body_location}')\n        \n        # Load image\n        image_data = await file.read()\n        image = Image.open(BytesIO(image_data))\n        \n        # Select prompt\n        prompt_template, prompt_type = select_prompt(\n            fitzpatrick_type, fever, rapidly_progressive, body_location, chief_complaint\n        )\n        \n        # Fill patient data\n        prompt = prompt_template.format(\n            age=age,\n            sex=sex,\n            fitzpatrick_type=fitzpatrick_type,\n            body_location=body_location,\n            duration=duration,\n            symptoms=symptoms or 'Not specified',\n            itch_score=itch_score,\n            pain_present='Yes' if pain_present else 'No',\n            warmth_present='Yes' if warmth_present else 'No',\n            systemic_symptoms=f\"Fever: {'Yes' if fever else 'No'}\",\n            recent_medications=recent_medications,\n            known_allergies=known_allergies,\n            medical_history=medical_history,\n            family_history=family_history,\n            recent_travel=recent_travel\n        )\n        \n        # Create messages\n        messages = [{\n            'role': 'user',\n            'content': [\n                {'type': 'image', 'image': image},\n                {'type': 'text', 'text': prompt}\n            ]\n        }]\n        \n        # Generate\n        print('🔮 Analyzing (DETERMINISTIC)...')\n        \n        inputs = processor.apply_chat_template(\n            messages,\n            add_generation_prompt=True,\n            tokenize=True,\n            return_dict=True,\n            return_tensors='pt'\n        ).to(model.device, dtype=torch.bfloat16)\n        \n        input_len = inputs['input_ids'].shape[-1]\n        \n        with torch.inference_mode():\n            generation = model.generate(\n                **inputs,\n                max_new_tokens=2048,\n                do_sample=False  # ✅ DETERMINISTIC\n            )\n        \n        response = processor.decode(generation[0][input_len:], skip_special_tokens=True)\n        \n        print('✅ Complete!')\n        \n        return {\n            'success': True,\n            'diagnosis': response,\n            'metadata': {\n                'prompt_used': prompt_type.value,\n                'fitzpatrick_adjusted': fitzpatrick_type >= 4,\n                'generation_mode': 'deterministic'\n            }\n        }\n    except Exception as e:\n        print(f'❌ Error: {e}')\n        raise HTTPException(500, str(e))\n\n# ═══════════════════════════════════════════════════════════\n# START SERVER\n# ═══════════════════════════════════════════════════════════\n\nngrok.set_auth_token('38NLQFj9JhH9qi5X9YxIURON0O4_45XszXGADUeqdKturWZSj')\npublic_url = ngrok.connect(8000)\n\nprint('\\n' + '='*70)\nprint('🌐 DERMACHECK AI CLINICAL v4 READY')\nprint('='*70)\nprint(f'📍 API URL: {public_url}')\nprint('='*70)\nprint('\\n✨ FEATURES:')\nprint('   ✅ 4 Specialized Prompts')\nprint('   ✅ Intelligent Selection')\nprint('   ✅ Fitzpatrick-Aware')\nprint('   ✅ Deterministic Generation')\nprint('\\n🚀 Starting server...')\n\nconfig = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info')\nserver = uvicorn.Server(config)\nawait server.serve()

## 🧪 Testing\n\nAfter server starts, copy the ngrok URL and test with:\n\n```python\nimport requests\n\nurl = 'YOUR_NGROK_URL/analyze'\n\nfiles = {'file': open('test_image.jpg', 'rb')}\ndata = {\n    'age': 45,\n    'sex': 'female',\n    'fitzpatrick_type': 5,\n    'body_location': 'sole of foot',\n    'duration': '3 months',\n    'chief_complaint': 'Dark spot on foot',\n    'symptoms': 'None',\n    'itch_score': 0,\n    'fever': False,\n    'rapidly_progressive': False\n}\n\nresponse = requests.post(url, files=files, data=data)\nprint(response.json())\n```\n\nExpected: Should trigger **melanoma_screening** prompt (acral location + Fitz V)!